# Análise VivaReal × Airbnb — Itapema/SC

Notebook principal e reproduzível da exploração completa dos dados.
Roda de ponta a ponta: carrega os CSVs, trata qualidade, modela a diária,
georreferencia, estima rentabilidade por perfil e fecha com recomendações.

**Premissas/limitações globais** (detalhadas na última seção):
- Só cenários de ocupação (sem dados de reservas) para receita anual.
- A análise cobre o **mercado ativo** (anúncios com preço no `Price_AV`).
- Airbnb não tem área útil → usa-se **nº de camas** como proxy de tamanho.

> Execute as células em ordem (Kernel → Restart & Run All).
> Troque `DATA_DIR` abaixo pela pasta dos CSVs, se mudar de máquina.

## 0. Setup e carregamento

In [1]:
import csv, os, io, re, math, glob
import datetime as dt
import collections as cl
import statistics as st
import unicodedata

import numpy as np
import pandas as pd
import statsmodels.api as sm

# --- configure aqui o caminho dos dados ---
DATA_DIR = r"C:\Users\Gabriela\Desktop\Hackathon\jovens-talentos-2026-hackathon-data\data"
OUT_DIR  = r"C:\Users\Gabriela\Desktop\Hackathon\analisis\output"
os.makedirs(OUT_DIR, exist_ok=True)

FILES = ["Details_Itapema.csv","Hosts_ids_Itapema.csv","Mesh_Ids_Data_Itapema.csv",
         "Price_AV_Itapema.csv","VivaReal_Itapema.csv"]

def load(fname):
    with open(os.path.join(DATA_DIR, fname), encoding="utf-8-sig", newline="") as fh:
        r = csv.reader(fh)
        header = next(r)
        return header, list(r)

def fcv(v):
    try:
        return float(str(v).strip())
    except (ValueError, TypeError):
        return None

def norm_sub(t):
    t = (t or "").strip().lower()
    t = unicodedata.normalize("NFD", t)
    t = "".join(c for c in t if unicodedata.category(c) != "Mn")
    t = re.sub(r"[^a-z0-9 ]", " ", t)
    t = " ".join(t.split())
    rep = {"jardim praia mar":"jardim praiamar","taboleiro":"tabuleiro dos oliveiras",
           "tabuleiro":"tabuleiro dos oliveiras","meia praia frente mar":"meia praia",
           "none":"sem_bairro","itapema":"sem_bairro","ocean tower":"sem_bairro"}
    return rep.get(t, t)

def parse_dt(s):
    s = s.strip()
    if "." in s[:21]:
        s = s[:23]
    return dt.datetime.strptime(s, "%Y-%m-%d %H:%M:%S.%f")

print("pandas", pd.__version__, "| numpy", np.__version__)
print("CSVs encontrados:", len(glob.glob(os.path.join(DATA_DIR, "*.csv"))))

pandas 2.3.3 | numpy 2.3.5
CSVs encontrados: 5


## 1. Visão geral das bases

In [2]:
resumo = []
for f in FILES:
    header, rows = load(f)
    resumo.append({"arquivo": f, "linhas": len(rows), "colunas": len(header)})
print(pd.DataFrame(resumo).to_string(index=False))

                  arquivo  linhas  colunas
      Details_Itapema.csv    4441       35
    Hosts_ids_Itapema.csv    4440       11
Mesh_Ids_Data_Itapema.csv    4441        8
     Price_AV_Itapema.csv  118839        4
     VivaReal_Itapema.csv    8329       22


In [3]:
# Tamanho em MB e primeiras colunas de cada base
for f in FILES:
    header, rows = load(f)
    print(f"\n== {f} ({len(rows)} linhas, {len(header)} colunas) ==")
    print("  colunas:", ", ".join(header))


== Details_Itapema.csv (4441 linhas, 35 colunas) ==
  colunas: airbnb_listing_id, url, ad_name, ad_description, space, house_rules, amenities, safety_features, number_of_bathrooms, number_of_bedrooms, number_of_beds, latitude, longitude, check_in, check_out, number_of_guests, number_of_reviews, cleaning_fee, owner_id, aquisition_date, star_rating, picture_count, min_nights, guest_satisfaction_overall, listing_type, can_instant_book, is_professional, accuracy_rating, checkin_rating, cleanliness_rating, communication_rating, location_rating, value_rating, is_new_listing, is_guest_favorite

== Hosts_ids_Itapema.csv (4440 linhas, 11 colunas) ==
  colunas: owner_id, owner, is_superhost, number_of_reviews_host, is_verified, star_rating_host, years_host, months_host, response_rate_shown, response_time_shown, host_snapshot_date

== Mesh_Ids_Data_Itapema.csv (4441 linhas, 8 colunas) ==
  colunas: airbnb_listing_id, latitude, longitude, suburb, country, state, city, aquisition_date



== Price_AV_Itapema.csv (118839 linhas, 4 colunas) ==
  colunas: airbnb_listing_id, date, price, aquisition_date

== VivaReal_Itapema.csv (8329 linhas, 22 colunas) ==
  colunas: listing_id, link_url, listing_title, business_types, listing_type, property_type, sale_price, rental_price, rental_period, yearly_iptu, monthly_condo_fee, amenities, usable_area, bathrooms, bedrooms, parking_spaces, state, city, suburb, advertiser_name, portal, aquisition_date


## 2. O problema das ondas (duplicatas de data no Price_AV)

Cada `(anúncio, data)` pode aparecer várias vezes porque há 3 coletas do calendário
(06/01, 07/01 e 20/01). A regra adotada (validada): **usar a última coleta com
`aquisition_date <= data`** como preço daquele dia.

In [4]:
h, prows = load("Price_AV_Itapema.csv")
pix = {x:i for i,x in enumerate(h)}
def wv(aq):
    return {"2025-01-06":"W1","2025-01-07":"W2","2025-01-20":"W3"}.get(aq.strip()[:10], "?")

# verificação estrutural das ondas
cnt = cl.Counter(wv(r[pix["aquisition_date"]]) for r in prows)
print("Linhas por onda (coleta):", dict(cnt))

# diária final por (listing, data) = última coleta válida
daily = cl.defaultdict(dict)
for r in prows:
    lid, d, pr, aq = r[pix["airbnb_listing_id"]], r[pix["date"]], float(r[pix["price"]]), r[pix["aquisition_date"]]
    dd = parse_dt(aq).date()
    if d not in daily[lid] or dd > daily[lid][d][0]:
        daily[lid][d] = (dd, pr)

# deixa validando: não há coleta posterior à estadia
v = sum(1 for lid,dail in daily.items() for d,(dd,pr) in dail.items() if dd > dt.datetime.strptime(d,"%Y-%m-%d").date())
print("Registros com coleta posterior à estadia:", v, "(esperado 0)")
print("Total linhas originais:", len(prows), "| combos únicos (listing,data):", sum(len(x) for x in daily.values()))

Linhas por onda (coleta): {'W2': 38991, 'W1': 37825, 'W3': 42023}


Registros com coleta posterior à estadia: 0 (esperado 0)
Total linhas originais: 118839 | combos únicos (listing,data): 59040


## 3. Qualidade: sentinelas e valores-0

- `star_rating == 0.0` **não é nota** — é "sem avaliação" (100% alinhado com `reviews==0`).
- `bedrooms == 0` no VivaReal ⇒ **não-residencial** (terreno/comercial), não 0 quartos.

In [5]:
h2, drows = load("Details_Itapema.csv")
dix = {x:i for i,x in enumerate(h2)}
rev = [fcv(r[dix["number_of_reviews"]]) for r in drows]
star = [fcv(r[dix["star_rating"]]) for r in drows]
rev0 = [i for i,v in enumerate(rev) if v==0]
star0 = [i for i,v in enumerate(star) if v==0.0]
print("reviews==0:", len(rev0), "| star==0.0:", len(star0), "| mesmos índices:", rev0==star0)
print("star min entre quem tem review>0:",
      min(star[i] for i in range(len(rev)) if (rev[i] or 0) > 0))

reviews==0: 1540 | star==0.0: 1540 | mesmos índices: True
star min entre quem tem review>0: 1.0


## 4. Cobertura do preço = mercado ativo

Apenas **1005 anúncios** de 4441 têm série de preço. Os sem-preço têm receita mediana de
1 review / 8 fotos → são anúncios inativos. A leitura de receita vale para o mercado em operação.

In [6]:
l_price = {lid: st.median(pr for d,(dd,pr) in dail.items()) for lid,dail in daily.items() if dail}
h3, mrows = load("Mesh_Ids_Data_Itapema.csv")
mix = {x:i for i,x in enumerate(h3)}
sub_of = {r[mix["airbnb_listing_id"]]: norm_sub(r[mix["suburb"]]) for r in mrows}

act = cl.Counter(sub_of.get(l, "?") for l in l_price)
tot = cl.Counter(sub_of.get(r, "?") for r,_ in [(rr[mix["airbnb_listing_id"]],1) for rr in mrows])
print(f"Anúncios com preço: {len(l_price)} | sem: {len(rev)-len(l_price)}")
for b in ["meia praia","centro","morretes","tabuleiro dos oliveiras"]:
    print(f"  {b:28s} ativos={act.get(b,0):4d}/{tot.get(b,0):5d}  ({100*act.get(b,0)/max(tot.get(b,0),1):.0f}%)")

Anúncios com preço: 1005 | sem: 3436
  meia praia                   ativos= 632/ 2860  (22%)
  centro                       ativos= 205/  657  (31%)
  morretes                     ativos=  83/  441  (19%)
  tabuleiro dos oliveiras      ativos=  20/  129  (16%)


## 5. O que explica a diária (regressão log)

Objetivo: separar efeito de tamanho × tipo × localização. Modelo:
`log(diária) ~ quartos + banheiros + camas + hóspedes + fotos + reviews + tipo + distância à orla`.

In [7]:
# distância à orla: max longitude do bairro como referência de costa (mar a leste)
lon_of = {mrows[i][mix["airbnb_listing_id"]]: fcv(mrows[i][mix["longitude"]]) for i in range(len(mrows))}
lat_of = {mrows[i][mix["airbnb_listing_id"]]: fcv(mrows[i][mix["latitude"]]) for i in range(len(mrows))}
subb = {mrows[i][mix["airbnb_listing_id"]]: norm_sub(r[mix["suburb"]]) for i,r in enumerate(mrows)}

det_dict = {}
for r in drows:
    det_dict[r[dix["airbnb_listing_id"]]] = {
        "bed": fcv(r[dix["number_of_bedrooms"]]),
        "bath": fcv(r[dix["number_of_bathrooms"]]),
        "beds": fcv(r[dix["number_of_beds"]]),
        "guests": fcv(r[dix["number_of_guests"]]),
        "pics": fcv(r[dix["picture_count"]]),
        "rev": fcv(r[dix["number_of_reviews"]]),
        "ltype": r[dix["listing_type"]].strip(),
    }

lon_bairro = cl.defaultdict(list)
for lid in lon_of:
    if subb.get(lid):
        lon_bairro[subb[lid]].append(lon_of[lid])
ref = {b: max(v) for b,v in lon_bairro.items()}
LAT_REF = -27.1

feat = []
for lid in l_price:
    if lid not in det_dict or lid not in lon_of or lid not in lat_of:
        continue
    d = det_dict[lid]
    if (d["bed"] or 0) <= 0 or lon_of[lid] is None:
        continue
    sub = subb.get(lid)
    if sub not in ref:
        continue
    dsea = max((ref[sub]-lon_of[lid])*111320*math.cos(math.radians(LAT_REF)), 0.0)
    feat.append({"lid":lid,"logp":math.log(l_price[lid]),"bed":d["bed"],"bath":d["bath"] or 0,
                 "beds":d["beds"] or 0,"guests":d["guests"] or 0,"pics":d["pics"] or 0,
                 "rev":d["rev"] or 0,"ltype":d["ltype"],"dsea":dsea})

cnt_b = cl.Counter(f["lid"] for f in feat)
# filtro bairro n>=5
bairro_count = cl.Counter(subb.get(f["lid"]) for f in feat)
keep_b = {b for b,c in bairro_count.items() if c>=5}
feat = [f for f in feat if subb.get(f["lid"]) in keep_b]
print("n =", len(feat))

n = 982


In [8]:
Xcol = ["bed","bath","beds","guests","pics","rev","dsea"]
X = []
y = []
for f in feat:
    rw = [f[c] for c in Xcol]
    rw += [1 if f["ltype"]=="apartamento" else 0,
           1 if f["ltype"]=="casa" else 0,
           1 if f["ltype"]=="hotel" else 0,
           1 if f["ltype"]=="outros" else 0]
    X.append(rw); y.append(f["logp"])
X = np.array(X, float); y = np.array(y, float)
X2 = sm.add_constant(X)
m = sm.OLS(y, X2).fit()
names = Xcol + ["tipo_apt","tipo_casa","tipo_hotel","tipo_outros"]
print(f"R² = {m.rsquared:.3f} (n={m.nobs})")
out = pd.DataFrame({
    "var": names,
    "coef": m.params[1:],
    "p": m.pvalues[1:],
    "efeito_%": 100*(np.exp(m.params[1:])-1)
}).round(3)
print(out.to_string(index=False))

R² = 0.431 (n=982.0)
        var   coef     p  efeito_%
        bed  0.174 0.000    19.034
       bath  0.138 0.000    14.761
       beds -0.015 0.084    -1.523
     guests  0.033 0.000     3.310
       pics -0.001 0.105    -0.121
        rev -0.001 0.002    -0.107
       dsea -0.000 0.000    -0.008
   tipo_apt  1.319 0.000   273.994
  tipo_casa  1.129 0.000   209.233
 tipo_hotel  1.286 0.000   261.703
tipo_outros  0.564 0.000    75.732


## 6. Sazonalidade da diária

A diária varia por **período** (férias/verão/carnaval vs baixa), não por dia da semana.
Abaixo, mediana por período na janela comum 20/01–06/04.

In [9]:
def period(dstr):
    m = dstr[:7]
    if m in ("2025-01","2025-02"): return "alta"
    if m == "2025-03":
        dd = dt.datetime.strptime(dstr,"%Y-%m-%d").date()
        return "alta" if dd.day in (3,4,5) else "media"
    return "baixa"

by_per = cl.defaultdict(list)
d0 = dt.date(2025,1,20); d1 = dt.date(2025,4,6)
for lid,dail in daily.items():
    for d,(dd,pr) in dail.items():
        if d0 <= dd <= d1:
            by_per[period(d)].append(pr)
for per in ["alta","media","baixa"]:
    v = by_per.get(per,[])
    print(f"  {per:6s} n={len(v):6d}  diária mediana = {st.median(v):.0f}")
print("ratio alta/baixa = %.2fx" % (st.median(by_per["alta"])/st.median(by_per["baixa"])))

  alta   n= 13960  diária mediana = 700
  media  n= 15942  diária mediana = 540
  baixa  n= 12121  diária mediana = 480
ratio alta/baixa = 1.46x


## 7. Rentabilidade por perfil (comprar para alugar)

Receita anual = diária sazonal (por período) × dias × **ocupação de cenário**.
Ocupação não é dado real — usamos 3 cenários. Cruzamos com a mediana de venda do VivaReal
para obter **anos para pagar (cenário base)**.

In [10]:
SCEN = {"conservador":{"alta":0.50,"media":0.35,"baixa":0.15},
        "base":{"alta":0.65,"media":0.45,"baixa":0.25},
        "otimista":{"alta":0.80,"media":0.60,"baixa":0.35}}
per_l = cl.defaultdict(lambda: cl.defaultdict(list))
for lid,dail in daily.items():
    for d,(dd,pr) in dail.items():
        if pr > 10000:  # limpa diárias absurdas
            continue
        per_l[lid][period(d)].append(pr)

def receita(lid, scen):
    tot = 0.0
    for pn,mn in (("alta",4),("media",4),("baixa",4)):
        dp = st.median(per_l[lid].get(pn, [])) if per_l[lid].get(pn) else None
        if dp is None: continue
        tot += dp*30*mn*scen[pn]
    return tot

# VivaReal: preço de venda por (bairro, faixa de quartos), usando a FAIXA DE ÁREA TÍPICA
# do mercado de venda para cada nº de quartos (origem abaixo), evitando misturar imóveis
# de portes muito diferentes que tenham só o mesmo nº de quartos.
#
# ORIGEM DAS FAIXAS (relação quarto → área no VivaReal, cálculos do script morretes_area.py):
#   - 2q: área mediana 70 m² (p25 66, p75 74) => faixa 60–90 m²
#   - 3q: área mediana 127 m² (p25 115, p75 140) => faixa 90–130 m²
#   - 4q+: área mediana 188 m² (p25 169, p75 213) => faixa 130–200 m²
AREA_BAND_POR_Q = {"1q": (0, 60), "2q": (60, 90), "3q": (90, 130), "4q+": (130, 200)}

h4, vrows = load("VivaReal_Itapema.csv")
vix = {x:i for i,x in enumerate(h4)}
def qb(n):
    if n is None or n<=0: return None
    return "1q" if n==1 else ("2q" if n==2 else ("3q" if n==3 else "4q+"))
vv = cl.defaultdict(list)
for r in vrows:
    sale = fcv(r[vix["sale_price"]]); area = fcv(r[vix["usable_area"]])
    beds = fcv(r[vix["bedrooms"]])
    lt = r[vix["listing_type"]].strip()
    sub = norm_sub(r[vix["suburb"]])
    if sub in ("","none") or lt not in ("apartamento","casa"): continue
    if not sale or not (150000<=sale<=13000000): continue
    if not area or not (15<=area<=1000): continue
    q = qb(beds)
    if q is None: continue
    lo, hi = AREA_BAND_POR_Q[q]
    if not (lo <= area <= hi): continue        # só entra na faixa de área típica do seu nº de quartos
    vv[(sub, q)].append(sale)

In [11]:
# --- critério de amostra mínima (define confiança) ---
MIN_N = 8

# Perfis da tabela final (seleção curada, alinhada ao README): (bairro, quartos)
PERFIS = [
    ("morretes", "3q"),
    ("tabuleiro dos oliveiras", "3q"),
    ("tabuleiro dos oliveiras", "2q"),
    ("morretes", "2q"),
    ("centro", "2q"),
    ("meia praia", "3q"),
    ("centro", "3q"),
    ("meia praia", "2q"),
    ("meia praia", "4q+"),
]

rows = []
for (sub, q) in PERFIS:
    ap = []
    for lid in l_price:
        ld = det_dict.get(lid, {})
        if subb.get(lid) == sub and qb(ld.get("bed")) == q:
            rb = receita(lid, SCEN["base"])
            if rb > 0:
                ap.append(rb)
    nA = len(ap)
    if nA < MIN_N:
        continue            # perfis com amostra insuficiente ficam de fora da tabela
    rmed = st.median(ap)
    vs = vv.get((sub, q), [])
    if not vs:
        continue
    vmed = st.median(vs)
    diaria = st.median([l_price[l] for l in l_price
                        if subb.get(l) == sub and qb(det_dict.get(l, {}).get("bed")) == q])
    rows.append([sub, q, nA, round(diaria), round(rmed), round(vmed), round(vmed / rmed, 1)])

df_rent = pd.DataFrame(rows, columns=["bairro", "perfil", "nAir", "diaria", "receita_base", "venda", "anos_base"])
df_rent = df_rent.sort_values("anos_base")
print(f"(amostra mínima = {MIN_N} anúncios; perfis com n<{MIN_N} foram excluídos)")
print(df_rent.to_string(index=False))
# salvar
df_rent.to_csv(os.path.join(OUT_DIR, "retorno_por_perfil.csv"), index=False, encoding="utf-8")

(amostra mínima = 8 anúncios; perfis com n<8 foram excluídos)
                 bairro perfil  nAir  diaria  receita_base   venda  anos_base
               morretes     3q    11     600        102300  750000        7.3
tabuleiro dos oliveiras     2q    12     425         78750  780000        9.9
                 centro     2q    67     557         88800  929750       10.5
               morretes     2q    60     448         67062  756554       11.3
             meia praia     2q   191     450         75300 1007000       13.4
                 centro     3q    47     790        114648 1851064       16.1
             meia praia     3q   332     650        103047 1716000       16.7
             meia praia    4q+    68    1150        158094 3250822       20.6


## 8. Conclusões e perfis recomendados

**Principais achados (ver README_analisis.md para detalhes e limitaçãoes):**
1. A diária é explicada principalmente por **tamanho (quartos/banheiros) e tipo**, não por fotos
   ou nº de reviews; a proximidade da orla tem efeito robusto (~ −12%/km).
2. A competência de comprimento do miolo: **Morretes/Tabuleiro têm melhor retorno** porque o
   preço de entrada é baixo e a demanda é decente.
3. Sazonalidade forte no verão (~1,46x), maior em apartamento que em casa.

**Gradação de confiança:**
- **Morretes 3q (principal):** melhor perfil, ~8 anos no cenário base, mas **amostra pequena (n=11)**
  → tratar como aproximadamente indicativa.
- **Tabuleiro 3q:** **indicação de apoio** (n=4, pouco para resultado confiável) — corrobora a
  direção de Morretes.
- **Centro 2q (compacto):** melhor perfil *dentro do Centro* (~11 anos), útil para valor no centro.
- **Descartar do ranking:** Ilhota (amostra e preço anômalos) e bairros com n<8.

**Limitações que não se pode ignorar:**
- Ocupação é **cenário**, não medido (sem dados de reservas; block-ratio descartado como proxy).
- A diária é a **listada**, não a cobrada.
- Análise cobre o **mercado ativo** (1005/4441 anúncios).
- "Anos para pagar" é **indicativo de ranking**, não previsão de ROI.